In [12]:
# Audio cutting/preparation
from pydub import AudioSegment
import os
import numpy as np
import math

# Removes silence at the beginning of wav file
def remove_silence_start(wav_path, output_file, silence_threshold=-46):
    audio = AudioSegment.from_wav(wav_path)
    print("Cutting silence at the beginning of the audio file...")

    # Find index of first not silent sample
    start_index = next((i for i, x in enumerate(audio) if x.dBFS > silence_threshold), None)

    # Safety buffer for not cutting to much of the last valve sound
    safety_margin = 30

    if (start_index is not None) and (start_index > safety_margin):
        # Cut silence at the beginning of the audio
        print(f"Old audio length: {len(audio)} ms")
        audio = audio[start_index - safety_margin:]
        print(f"New audio length: {len(audio)} ms\n")
    else:
        print("No cutting required\n")

    # Save edited file
    audio.export(output_file, format="wav")

# Removes silence at the end of wav file
def remove_silence_end(wav_path, output_file, silence_threshold=-46):
    audio = AudioSegment.from_wav(wav_path)
    print("Cutting silence at the end of the audio file...")

    # Find index of last not silent sample
    end_index = next((len(audio) - 1 - i for i, x in enumerate(reversed(audio)) if x.dBFS > silence_threshold), None)

    # Safety buffer for not cutting to much of the last valve sound
    safety_buffer = 30

    if (end_index is not None) and ((end_index + safety_buffer) < (len(audio) - 1)):
        # Cut silence at the end of the audio
        print(f"Old audio length: {len(audio)} ms")
        audio = audio[:-(len(audio) - (end_index + safety_buffer) - 1)]
        print(f"New audio length: {len(audio)} ms\n")
    else:
        print("No cutting required\n")

    # Save edited file
    audio.export(output_file, format="wav")

# Determine the k-th percentile based on the proportion of valve noise in the audio file
def get_percentile_dbfs(wav_path, valve_time, cycle_duration, correction_factor=0.9):
    audio = AudioSegment.from_wav(wav_path)

    # Takes a sample out of the middle of the audio file for the percentile determination -> beginning/end of audio file could contain silence
    # samples = []
    # for i, x in enumerate(audio[len(audio)/2 - 4*(cycle_duration):len(audio)/2]):
    #     samples.append(x.dBFS)

    samples = []
    for i, x in enumerate(audio[len(audio)*0.25 - 4*(cycle_duration):len(audio)*0.25]):
        samples.append(x.dBFS)

    for i, x in enumerate(audio[len(audio)*0.75 - 4*(cycle_duration):len(audio)*0.75]):
        samples.append(x.dBFS)

    # Determines percentile
    noise_percentage = 100 - (100 * (valve_time / cycle_duration))
    percentile_dbfs = np.percentile(samples, noise_percentage)
    percentile_dbfs = math.trunc(percentile_dbfs)
    print(f"{noise_percentage}% of the audio file is quieter than {percentile_dbfs} dBFS")

    # Applying correction factor
    percentile_dbfs *= correction_factor
    print(f"The dBFS with a correction factor of {correction_factor} is {percentile_dbfs}\n")

    return percentile_dbfs

# Cut WAV file in equally long chunks
def split_wav(wav_path, output_directory, valve_time, cycle_duration):
    audio = AudioSegment.from_wav(wav_path)
    print("Splitting wav file into single valve sounds...")

    # Make sure that the output directory exists
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    quarter_valve_pause = (cycle_duration - valve_time) / 4
    total_duration = len(audio)
    chunk_number = 1
    current_position = 0

    while current_position < total_duration:
        end_position = current_position + cycle_duration
        if end_position > total_duration:
            end_position = total_duration

        chunk = audio[current_position:end_position]

        current_position = end_position

        chunk_length = len(chunk)
        # Fill last audio chunk with zeroes, if it's too short
        if chunk_length < cycle_duration:
            if chunk_length <= valve_time:
                break
            else:
                chunk += AudioSegment.silent(duration=(cycle_duration - len(chunk)), frame_rate=audio.frame_rate)

        output_file = os.path.join(output_directory, f"{os.path.basename(wav_path).split('_')[0]}_{cycle_duration}_{valve_time}_chunk_{chunk_number}.wav")
        chunk.export(output_file, format="wav") 
        chunk_number += 1
    
    print(f"{chunk_number - 1} valve sounds detected\n")

In [60]:
# Audio cutting/preparation

"""
Don't forget to change the machine parameters and rec name
"""
rec_name = 'rec66'
cycle_duration = 150
valve_time = 30
class_name = 'positive'
"""
Don't forget to change the machine parameters and rec name
"""

wav_path = os.path.join('../','./', 'audios', 'audios', f'{class_name}', f'{rec_name}.wav')
# wav_path = os.path.join('../','./', 'audios', 'audios', 'positive', f'{rec_name}.wav)


# Determine the k-th percentile based on the proportion of valve noise in the audio file
percentile_dbfs = get_percentile_dbfs(wav_path, valve_time, cycle_duration, correction_factor=0.86)

# For audio recorded with a dummy plug
# percentile_dbfs = get_percentile_dbfs(wav_path, valve_time, cycle_duration, correction_factor=0.70)

# Cutting silence at the beginning of the audio file
output_file = os.path.join(f"{os.path.splitext(wav_path)[0]}_removed_silence_beginning.wav")
remove_silence_start(wav_path, output_file, silence_threshold=percentile_dbfs)

# Cutting silence at the end of the audio file
wav_path = output_file
output_file = os.path.join(f"{os.path.splitext(wav_path)[0]}_and_end.wav")
remove_silence_end(wav_path, output_file, silence_threshold=percentile_dbfs)

# Cutting WAV file in equally long chunks
wav_path = output_file
output_directory = os.path.join('./', 'audio_segments', f'{class_name}')
split_wav(wav_path, output_directory, valve_time, cycle_duration)

80.0% of the audio file is quieter than -19 dBFS
The dBFS with a correction factor of 0.86 is -16.34

Cutting silence at the beginning of the audio file...
Old audio length: 29952 ms
New audio length: 28809 ms

Cutting silence at the end of the audio file...
No cutting required

Splitting wav file into single valve sounds...
192 valve sounds detected

